#### Data Loader
Copies one generated batch from the `dev.stepright.staging` volume into the
`dev.stepright.landing` volume, and verifies row counts per source.

This notebook stands in for Debezium and the warehouse's nightly export — the real
delivery mechanisms this project doesn't have running in this course. In production, 
these sources would land in the volume continuously and automatically.

**Before running:** generate a batch locally with `data_generator.py`, then manually
upload that batch's folder into `dev.stepright.staging/batch_<n>/` using the Databricks
workspace UI (Catalog Explorer → dev → stepright → staging → Upload).

In [0]:
%sql
-- One-time: the staging volume, separate from landing, only used to receive
-- manually-uploaded batch files before we load them.
CREATE VOLUME IF NOT EXISTS dev.stepright.staging;


In [0]:
dbutils.widgets.text("batch", "0", "Batch number to load")
batch_num = dbutils.widgets.get("batch")

staging_root = f"/Volumes/dev/stepright/staging/batch_{batch_num}"
landing_root = f"/Volumes/dev/stepright/landing"

print(f"Loading batch_{batch_num} from `{staging_root}` to `{landing_root}`")


#### Copy each source's files from staging into its landing subfolder

In [0]:
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

In [0]:
sub_folders = [
    "orders_cdc",
    "order_items_cdc",
    "customers_cdc",
    "products",
    "categories",
    "clickstream",
    "inventory"
]

for folder in sub_folders:
    src = f"{staging_root}/{folder}"
    dst = f"{landing_root}/{folder}"
    
    if dbutils.fs.ls(src):
        dbutils.fs.cp(src, dst, recurse=True)
        print(f"Copied: {src} -> {dst}")
    else:
        print(f"Skipped folder {folder} - nothing staged for batch_{batch_num}. ")

#### Verify row counts per source
This is also our environment health check — if Unity Catalog and the landing volume
are correctly wired, every source below should show a non-zero row count.

In [0]:
csv_sources = ["products", "categories", "inventory"]
json_sources = ["orders_cdc", "order_items_cdc", "customers_cdc", "clickstream"]
 
print(f"{'source':<20}{'format':<8}{'row_count'}")
print("-" * 45)
 
for folder in csv_sources:
    path = f"{landing_root}/{folder}"
    try:
        count = spark.read.option("header", "true").csv(path).count()
    except Exception:
        count = 0
    print(f"{folder:<20}{'csv':<8}{count}")
 
for folder in json_sources:
    path = f"{landing_root}/{folder}"
    try:
        count = spark.read.json(path).count()
    except Exception:
        count = 0
    print(f"{folder:<20}{'json':<8}{count}")